# 11.11 Time Series Forecasting Strategy

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.11_time-series-forecasting-strategy.ipynb)

Companion code for Section 11.11 of the book. Run the cells in order from top to bottom.

**Example 11.11.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

# ── Walk-forward validation implementation ────────────────────────
def walk_forward_validation(df, target, feature_cols, model,
                              min_train_size=180, test_size=30,
                              step_size=30, verbose=True):
    '''
    Walk-forward (expanding window) cross-validation for time series.
    Trains only on past data, evaluates on future data.
    '''
    results = []
    n = len(df)

    cutoffs = range(min_train_size, n - test_size, step_size)

    for i, cutoff in enumerate(cutoffs):
        train_df = df.iloc[:cutoff]
        test_df  = df.iloc[cutoff:cutoff+test_size]

        X_tr = train_df[feature_cols]
        y_tr = train_df[target]
        X_te = test_df[feature_cols]
        y_te = test_df[target]

        # Fit and predict
        from sklearn.base import clone
        m = clone(model)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        mae  = mean_absolute_error(y_te, y_pred)
        mape = np.mean(np.abs((y_te-y_pred)/y_te))*100

        results.append({
            'cutoff': df.index[cutoff],
            'train_size': cutoff,
            'test_start': test_df.index[0],
            'mae': mae,
            'mape': mape
        })

        if verbose:
            print(f'  Fold {i+1}: train={cutoff}  test={test_df.index[0].date()}  MAE={mae:.2f}  MAPE={mape:.2f}%')

    return pd.DataFrame(results)


# ── Build data and features ───────────────────────────────────────
np.random.seed(42)
n = 730
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)
y = 1000 + 0.8*t + 300*np.sin(2*np.pi*t/365-np.pi/2) + 150*np.sin(2*np.pi*t/7) + np.random.normal(0,60,n)

df = pd.DataFrame(index=dates)
df['y'] = y
df['dow_sin']  = np.sin(2*np.pi*dates.dayofweek/7)
df['doy_sin']  = np.sin(2*np.pi*dates.dayofyear/365.25)
df['doy_cos']  = np.cos(2*np.pi*dates.dayofyear/365.25)
df['trend']    = t
for lag in [1,7,14,28,90,365]:
    df[f'lag_{lag}'] = df['y'].shift(lag)
for w in [7,28]:
    df[f'roll_mean_{w}'] = df['y'].shift(1).rolling(w).mean()
df = df.dropna()

feature_cols = [c for c in df.columns if c != 'y']

# ── Run walk-forward validation ───────────────────────────────────
print('=== Walk-Forward Validation ===')
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
wf_results = walk_forward_validation(df, 'y', feature_cols, xgb_model,
                                       min_train_size=180, test_size=30, step_size=30)

print(f'\nWalk-forward summary:')
print(f'  Folds:     {len(wf_results)}')
print(f'  Mean MAE:  {wf_results.mae.mean():.2f} ± {wf_results.mae.std():.2f}')
print(f'  Mean MAPE: {wf_results.mape.mean():.2f}% ± {wf_results.mape.std():.2f}%')

# ── Multi-step forecasting strategies ────────────────────────────
print('\n=== Multi-Step Forecasting Strategies ===')

strategies = {
    'Recursive':  'Train 1-step model; recursively predict, feeding prediction back as lag feature',
    'Direct':     'Train separate model for each horizon h = 1,2,...,H (most accurate but slow)',
    'MIMO':       'Train single model predicting all H steps simultaneously (captures dependencies)',
    'DirRec':     'Combine: Direct models but use previous predictions as features',
}
for name, desc in strategies.items():
    print(f'\n  {name}:')
    print(f'    {desc}')

# Implement direct strategy (train separate model per horizon)
print('\nDirect strategy: MAE per horizon')
train_direct = df.iloc[:500]
test_direct  = df.iloc[500:]

for h in [1, 7, 14, 30]:
    # Create target shifted by h
    y_h = df['y'].shift(-h)
    valid = df.dropna().index.intersection(y_h.dropna().index)
    X_h   = df.loc[valid, feature_cols]
    y_h_v = y_h.loc[valid]

    X_tr_h = X_h[X_h.index <= train_direct.index[-1]]
    y_tr_h = y_h_v[y_h_v.index <= train_direct.index[-1]]
    X_te_h = X_h[X_h.index > train_direct.index[-1]]
    y_te_h = y_h_v[y_h_v.index > train_direct.index[-1]]

    if len(X_tr_h) > 10 and len(X_te_h) > 0:
        m_h = xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
        m_h.fit(X_tr_h, y_tr_h)
        pred_h = m_h.predict(X_te_h)
        mae_h  = mean_absolute_error(y_te_h, pred_h)
        mape_h = np.mean(np.abs((y_te_h-pred_h)/y_te_h))*100
        print(f'  h={h:>3}: MAE={mae_h:.2f}  MAPE={mape_h:.2f}%  (error grows with horizon)')

# ── Ensemble forecasting ─────────────────────────────────────────
print('\n=== Ensemble Forecasting ===')
X_tr = df.iloc[:600][feature_cols]; y_tr = df.iloc[:600]['y']
X_te = df.iloc[600:][feature_cols]; y_te = df.iloc[600:]['y']

ensemble_models = {
    'XGBoost':  xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1),
    'RF':       __import__('sklearn.ensemble', fromlist=['RandomForestRegressor']).RandomForestRegressor(100,
random_state=42, n_jobs=-1),
}

preds = {}
for name, model in ensemble_models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    preds[name] = pred
    mae = mean_absolute_error(y_te, pred)
    print(f'  {name:<12}: MAE={mae:.2f}')

# Simple average ensemble
ensemble_pred = np.mean(list(preds.values()), axis=0)
mae_ens = mean_absolute_error(y_te, ensemble_pred)
print(f'  Ensemble avg: MAE={mae_ens:.2f}  (usually best!)')